# 21 · Logging, Config & CLI

Production pipelines don't `print` — they **log**. They don't hard-code
secrets — they read **config** and **environment variables**. And they're
runnable from the command line with **arguments**. This notebook covers all
three.

## Logging instead of print

`logging` gives you levels (`DEBUG` < `INFO` < `WARNING` < `ERROR`), timestamps,
and the ability to route output to files or services — without changing your
code. Configure once; call `logger.info(...)` everywhere.

In [ ]:
import logging, sys

logger = logging.getLogger('pipeline')
logger.handlers.clear()
logger.setLevel(logging.INFO)
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter(
    '%(asctime)s | %(levelname)-7s | %(name)s | %(message)s',
    datefmt='%H:%M:%S'))
logger.addHandler(handler)

logger.debug('you will NOT see this (below INFO)')
logger.info('extract started')
logger.warning('3 rows had blank emails')
logger.error('failed to reach source API')

## Log structured context

Include machine-readable context (row counts, table names) so logs are
searchable. In real systems you'd emit JSON logs; here we show rich messages.

In [ ]:
rows, table = 1523, 'orders'
logger.info('loaded rows=%d table=%s', rows, table)   # %-style args, lazy

try:
    1 / 0
except ZeroDivisionError:
    logger.exception('transform crashed')   # logs the full traceback

## Configuration & environment variables

Secrets (DB passwords, API keys) and environment-specific settings belong in the
**environment**, not code. `os.environ` reads them; `python-dotenv` loads a
local `.env` file in development. Provide sensible defaults.

In [ ]:
import os

# In real code: from dotenv import load_dotenv; load_dotenv()
os.environ.setdefault('BATCH_SIZE', '1000')     # pretend it came from .env
os.environ.setdefault('ENV', 'dev')

batch_size = int(os.environ.get('BATCH_SIZE', '500'))
env = os.environ.get('ENV', 'dev')
db_url = os.environ.get('DATABASE_URL', 'sqlite:///data/retail.db')  # default
print(f'env={env} batch_size={batch_size}')
print('db_url:', db_url)

## A config object

Bundle settings into one typed object (a `dataclass` or pydantic
`BaseSettings`) so the rest of the code depends on a clean interface, not scattered
`os.environ` calls.

In [ ]:
import os
from dataclasses import dataclass

@dataclass(frozen=True)
class Config:
    env: str
    batch_size: int
    db_url: str

    @classmethod
    def from_env(cls):
        return cls(
            env=os.environ.get('ENV', 'dev'),
            batch_size=int(os.environ.get('BATCH_SIZE', '500')),
            db_url=os.environ.get('DATABASE_URL', 'sqlite:///data/retail.db'),
        )

cfg = Config.from_env()
print(cfg)

## Command-line arguments with `argparse`

A pipeline script should accept parameters (which date to run, dry-run, etc.).
`argparse` parses `sys.argv` and generates `--help` for free. Here we parse an
explicit list so it runs in a notebook; a real script omits the list and reads
the actual command line.

In [ ]:
import argparse

parser = argparse.ArgumentParser(description='Run the daily ETL')
parser.add_argument('--date', required=True, help='YYYY-MM-DD to process')
parser.add_argument('--dry-run', action='store_true', help='validate only')
parser.add_argument('--batch-size', type=int, default=1000)

args = parser.parse_args(['--date', '2024-06-01', '--dry-run'])
print('date:', args.date)
print('dry_run:', args.dry_run)
print('batch_size:', args.batch_size)

### Recap

Use `logging` (levels, formatters, `logger.exception`) instead of `print`; read
secrets/settings from the environment with `python-dotenv` and defaults; wrap
them in a typed `Config`; expose parameters via `argparse`. Next: testing with
pytest.